In [ ]:
import pprint

In [ ]:
from litellm import completion

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The unit of temperature.",
                    },
                },
                "required": ["location"],
            }
        }
    }
]

messages = [{"role": "user", "content": "What's the weather like in Boston today?"}]

response = completion(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",  # name can be anything locally
    api_base="http://localhost:8080/v1",
    api_key="not-needed",   # llama.cpp doesn't require it
    messages=messages,
    tools=tools,
)

print(response.choices[0].message.tool_calls[0].function)


In [ ]:
!pip install llama_index
!pip install llama_index.llms.litellm

In [ ]:
import nest_asyncio
from llama_index.llms.litellm import LiteLLM
from llama_index.core.agent.workflow import ReActAgent, AgentStream
from llama_index.core.tools import FunctionTool

nest_asyncio.apply()

def add(a:int|float, b:int|float) -> int|float:
    """Adds 2 integers or floats"""
    return a+b

llm = LiteLLM(
    model="openai/unsloth/gemma-4-E4B-it-GGUF:Q8_0",
    api_base="http://localhost:8080/v1",
    api_key="sk-no-key-required"
)

agent = ReActAgent(
    tools=[FunctionTool.from_defaults(add)],
    llm=llm
)
handler = agent.run("What is 21.921 + 23.21?")
async for event in handler.stream_events():
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)

In [ ]:
import httpx, os

key = os.getenv("Z_AI_API_KEY")

r = httpx.post(
    "https://api.z.ai/api/anthropic/v1/messages",
    headers={
        "x-api-key": key,
        "content-type": "application/json"
    },
    json={
        "model": "glm-5.1",
        "messages": [{"role": "user", "content": "test"}],
        "max_tokens": 10
    },
    timeout=30
)

print(r.status_code)
print(r.text)

In [ ]:
key = os.getenv("ZAI_API_KEY")
key

In [ ]:
import httpx, os

key = os.getenv("ZAI_API_KEY")

r = httpx.post(
    "https://api.z.ai/api/coding/paas/v4/chat/completions",
    headers={
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "glm-5.1",
        "messages": [
            {"role": "user", "content": "what is 1 + 1"}
        ],
        "max_tokens": 100
    },
    timeout=30
)

data = r.json()
msg = data["choices"][0]["message"]

print("Answer:", msg.get("content"))
print("Reasoning:", msg.get("reasoning_content"))

In [ ]:
# User-001-Key-1
# sk-6u8AG5V_riB7HNJL2LknCg

# User-002-Key-1
# sk-Pm5rFFZ9pwEgKDklGt87og

# User-003-Key-1
# sk-aorPyWzQfmJpUdQHQ3dOrA

In [33]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-my-super-secret-key",
    base_url="http://localhost:4000/v1"
)

response = client.chat.completions.create(
    model="local-gemma",
    messages=[
        {
            "role": "user",
            "content": "What is the temperature in Dhaka?"
        }
    ],
    tool_choice="auto"
)

print(response)

ChatCompletion(id='chatcmpl-2nJl6xgjASqgL29fSi75IkuzJYyTJMSL', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I do not have real-time access to current weather data, so I cannot tell you the exact temperature in Dhaka right now.\n\nTo find the current temperature, please check a reliable weather website or use a weather app!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, provider_specific_fields={'refusal': None}), provider_specific_fields={})], created=1778820743, model='local-gemma', object='chat.completion', service_tier=None, system_fingerprint='b9121-89730c8d2', usage=CompletionUsage(completion_tokens=46, prompt_tokens=16, total_tokens=62, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=2)), timings={'cache_n': 2, 'prompt_n': 14, 'prompt_ms': 150.651, 'prompt_per_token_ms': 10.760785714285715, 'prompt_per_second': 92

In [38]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-my-super-secret-key",
    base_url="http://localhost:4000"
)

response = client.responses.create(
    model="local-gemma",
    input="what is the current temperature of dhaka bangladesh",
    stream=True,
    tool_choice="auto",
    tools=[
        {
            "type": "mcp",
            "server_label": "litellm",
            "server_url": "http://localhost:4000/mcp/utility_server",
            # "server_url": "litellm_proxy/mcp/utility_server",
            "require_approval": "never"
        }
    ]
)

output_text = ""

for event in response:
    if event.type == "response.output_text.delta":
        output_text += event.delta
        print(event.delta, end="", flush=True)

print("\n\nFINAL:\n", output_text)

The current temperature in Dhaka, Bangladesh is 30.4°C with light drizzle.

FINAL:
 The current temperature in Dhaka, Bangladesh is 30.4°C with light drizzle.
